In [1]:
import pandas as pd
import numpy as np

RAW_PATH = '../../jhani_pandit_dataset/Fitness_Survey__Responses_.xlsx'
CLEAN_PATH = '../../jhani_pandit_dataset/Fitness_Survey__Responses_clean.xlsx'

df = pd.read_excel(RAW_PATH)
print('Размер исходного набора данных:', df.shape)

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [2]:
def norm_loc(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).strip().split()).casefold()

INDIA_LOCATIONS = {
    "Mumbai", "Palghar", "Navi mumbai", "Indore", "Vasai", "Bangalore",
    "Kolkata", "Vasai, palghar", "Calicut", "Thane", "Mumbai suburban",
    "Kolhapur", "Mumbay", "Pune", "Nanded", "Kalyan", "Bhopal",
    "Kandivali east, mumbai", "Nashik", "Bombay", "Bhandup",
    "Nagala park", "New delhi", "Kolhapye", "Kop", "Kohlapur",
    "Kohlapu", "Ahmedabad", "Jamnagar, gujarat", "Gujarat",
    "Bengaluru", "Virar", "Ichalkaranji", "Vasgade", "Bom",
    "Viman nagar", "Poona", "Vapi", "Gurugram", "Delhi",
    "Hyderabad", "Jalandhar", "Chennai", "Nellore"
}

NOT_INDIA = {
    "Nigeria, yaba,lagos.", "London", "Bucharest",
    "Singapore", "San francisco"
}

ERROR_VALUES = {"K", "M", "Ketan mehta"}

INDIA = {norm_loc(x) for x in INDIA_LOCATIONS}
NOT_IND = {norm_loc(x) for x in NOT_INDIA}
ERROR = {norm_loc(x) for x in ERROR_VALUES}

def map_country(location):
    loc = norm_loc(location)
    if loc in ERROR or loc == "":
        return "error"
    if loc in INDIA:
        return "India"
    if loc in NOT_IND:
        return "Not India"
    return "unknown"


df["Country"] = df["Location (city name or district name)"].apply(map_country)
df = df[df["Country"] == "India"].copy()

In [3]:
CITY_MAPPING = {
    'mumbai': 'Mumbai',
    'mumbay': 'Mumbai',
    'bombay': 'Mumbai',
    'bom': 'Mumbai',
    'mumbai suburban': 'Mumbai',
    'kandivali east, mumbai': 'Mumbai',
    'bhandup': 'Mumbai',

    'kolhapur': 'Kolhapur',
    'kohlapur': 'Kolhapur',
    'kohlapu': 'Kolhapur',
    'kolhapye': 'Kolhapur',
    'kop': 'Kolhapur',
    'nagala park': 'Kolhapur',

    'pune': 'Pune',
    'poona': 'Pune',
    'viman nagar': 'Pune',

    'vasai': 'Vasai',
    'vasai, palghar': 'Vasai',

    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',

    'new delhi': 'Delhi',
}

df["Location_norm"] = (
    df["Location (city name or district name)"]
    .apply(norm_loc)
    .replace("", np.nan)
    .map(lambda x: CITY_MAPPING.get(x, x.title()) if pd.notna(x) else x)
)

In [4]:
PLACE_COL = "Your general place of wellness/fitness routine is"

place_key = (
    df[PLACE_COL]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.casefold()
)

PLACE_MAPPING = {
    "home - without equipment": "Home - without equipment",
    "home - with equipment": "Home - with equipment",

    "gym / studio": "Gym",
    "sports club": "Gym",
    "sports": "Gym",

    "park": "Outdoor",
    "parks": "Outdoor",
    "garden": "Outdoor",
    "outdoor": "Outdoor",
    "outdoors": "Outdoor",
    "walking outdoor": "Outdoor",
    "walk swimming": "Outdoor",

    "hybrid - gym and at home": "Hybrid - gym and at home",
}

df["Place of wellness/fitness routine"] = place_key.map(PLACE_MAPPING)

In [5]:
ROUTINE_COL = "What would best describe a wellness/fitness routine you follow or are likely to follow?"

routine_clean = (
    df[ROUTINE_COL]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

ROUTINE_MAPPING = {
    "Light physical activity (running, push-ups, sit-ups etc.)": "Light physical activity",
    "Swimming & cycling": "Light physical activity",
    "Badminton": "Light physical activity",
    "Walking & Stretching excercise": "Light physical activity",
    "Walking for an hour": "Light physical activity",

    "Rigorous physical exercise with equipments (e.g. weight training, cross-fit etc.)": "Rigorous physical exercise with equipments",
    "HIIT exercises from YouTube daily": "Rigorous physical exercise with equipments",

    "Yoga & breathing exercises": "Yoga",
    "Physical Exercise + Yoga": "Yoga",

    "Meditation/Mindfulness practice": "Meditation/mindfulness practice",

    "I don't follow any wellness/fitness routine": "I don't follow any wellness/fitness routine",
}

df["Wellness/fitness_routine"] = routine_clean.map(ROUTINE_MAPPING)

In [6]:
SUBSCRIPTION_COL = (
    "If you were to subscribe such an online  wellness/fitness training service, "
    "what would describe the ideal subscription cycle for you?"
)

sub_clean = (
    df[SUBSCRIPTION_COL]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

SUBSCRIPTION_MAPPING = {
    "Annual subscription with price discount benefits": "Annual subscription",
    "Monthly subscription at a price slightly higher than annual subscription": "Monthly subscription",
    "Weekly subscription at a price slightly higher than monthly subscription": "Weekly subscription",
    "Session based subscription - i.e. top up with a number of sessions with flexibility to use those over a period of time at a higher price": "Session-based subscription",
    "Without clear advantage, don't see a point in paying.": "Other",
    "Not interested - so far outdoor is working fine": "Other",
    "No": "Other",
    "Half yearly": "Other",
}

df["Subscription_cycle"] = sub_clean.map(SUBSCRIPTION_MAPPING)


In [7]:
BARRIERS_COL = (
    "Which of the followings would best describe the main challenge you face "
    "that restricts you from following a regular wellness/fitness routine? "
    "(you can choose more than one option)"
)

BARRIER_MAPPING = {
    "Lack of motivation / time": "Lack of motivation / time (routine)",
    "Want a personal trainer but can't afford": "Can't afford personal trainer (routine)",
    "Can't afford to spend on gym/classes subscription": "Can't afford gym/classes (routine)",
    "Can't find a good trainer/expert practitioner": "Can't find trainer/expert (routine)",
    "Unavailability of a good gym/classes nearby": "No gym/classes nearby (routine)",
    "Don't know where and how to start": "Don't know how to start (routine)",
    "I follow a regular wellness/fitness routine": "No barriers (routine)",
    "Maintaining a proper diet": "Other (routine)",
    "Diet": "Other (routine)",
    "Lack of time is all": "Other (routine)",
    "Walking an hour": "Other (routine)",
    "Lack of yoga center...so prefer walking atleast 10000 steps .": "Other (routine)",
}


df["Barriers_routine"] = (
    df[BARRIERS_COL]
    .fillna("")
    .astype("string")
    .str.split(",")
    .apply(lambda xs: ", ".join(sorted({
        BARRIER_MAPPING.get(x.strip(), "Other (routine)")
        for x in xs if x.strip()
    })))
)


barrier_dummies = (
    df["Barriers_routine"]
    .str.get_dummies(sep=", ")
    .astype(int)
)

if "No barriers (routine)" in barrier_dummies.columns:
    other_cols = barrier_dummies.columns.drop("No barriers (routine)")
    has_other = barrier_dummies[other_cols].sum(axis=1) > 0
    barrier_dummies.loc[has_other, "No barriers (routine)"] = 0

df = pd.concat([df, barrier_dummies], axis=1)

In [8]:
DIET_BARRIERS_COL = (
    "Which of the followings would best describe the main challenge(s) you face that restrict you from following a well-balanced nutrition diet routine? "
    "(you can choose more than one option)"
)

DIET_BARRIER_MAPPING = {
    "Find it too difficult to follow a routine": "Too difficult (diet)",
    "Temptation for junk food": "Junk food temptation (diet)",
    "My schedule doesn't permit regularity in my eating habits": "Schedule issues (diet)",
    "Lack of understanding what to eat when and in what proportion": "Don't know what/how much to eat (diet)",
    "Can't cook at home": "Can't cook at home (diet)",

    "I follow a balanced diet routine": "No barriers (diet)",

    "I don't care enough": "Other (diet)",
}

df["Barriers_diet"] = (
    df[DIET_BARRIERS_COL]
    .fillna("")
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.split(",")
    .apply(lambda xs: ", ".join(sorted({
        DIET_BARRIER_MAPPING.get(x.strip(), "Other (diet)")
        for x in xs if x.strip()
    })))
)

diet_dummies = (
    df["Barriers_diet"]
    .str.get_dummies(sep=", ")
    .astype(int)
)

if "No barriers (diet)" in diet_dummies.columns:
    other_cols = diet_dummies.columns.drop("No barriers (diet)")
    has_other = diet_dummies[other_cols].sum(axis=1) > 0
    diet_dummies.loc[has_other, "No barriers (diet)"] = 0

df = pd.concat([df, diet_dummies], axis=1)

In [9]:
ONLINE_BARRIERS_COL = (
    "What would best describe the challenge(s) to use an online wellness/fitness service as a mainstream method to achieve your wellness/fitness goals? "
    "(you can choose more than one option)"
)

ONLINE_BARRIER_MAPPING = {
    "Absence of ambience of a gym/studio and fellow members in online mode": "No ambience (online)",
    "I don’t think online training is effective": "Not effective (online)",
    "I don't want to invest in buying any accessories at home": "No accessories (online)",
    "Instant feedback is very important to me which I think online delivery wellness/fitness services lack": "Need feedback (online)",
    "I don't have access to right kind of space to practice regularly": "No space (online)",
}

df["Barriers_online"] = (
    df[ONLINE_BARRIERS_COL]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .apply(lambda t: (
        ", ".join(sorted({
            ONLINE_BARRIER_MAPPING[k]
            for k in ONLINE_BARRIER_MAPPING
            if (not pd.isna(t)) and (k in t)
        })) or ("Other (online)" if (not pd.isna(t)) and t else "")
    ))
    .replace({"": np.nan})
)

online_dummies = (
    df["Barriers_online"]
    .fillna("")
    .str.get_dummies(sep=", ")
    .astype(int)
)

df = pd.concat([df, online_dummies], axis=1)

In [10]:
missing_df = df.isna().sum()
missing_df = missing_df[missing_df > 0]
print('Колонки с пропусками:')
print(missing_df)
print()
print('Количество дублирующихся строк:', df[df.duplicated(keep=False)].shape[0])

Колонки с пропусками:
What would best describe the challenge(s) to use an online wellness/fitness service as a mainstream method to achieve your wellness/fitness goals? (you can choose more than one option)    5
If you were to subscribe such an online  wellness/fitness training service, what would describe the ideal subscription cycle for you?                                                       6
Subscription_cycle                                                                                                                                                                          6
Barriers_online                                                                                                                                                                             5
dtype: int64

Количество дублирующихся строк: 0


In [11]:
df.to_excel(CLEAN_PATH, index=False)
print(f'Сохранено: {CLEAN_PATH}')
print('Финальный размер:', df.shape)

Сохранено: ../../jhani_pandit_dataset/Fitness_Survey__Responses_clean.xlsx
Финальный размер: (235, 52)
